In [15]:
import pandas as pd
# 1. Load the dataset

input_file = "groceries.csv"

# The dataset does not have a proper header.
# Each row represents one transaction.
df = pd.read_csv(input_file, header=None)

print("Original shape:", df.shape)

# 2. Remove completely empty rows and columns

df = df.dropna(axis=0, how="all")
df = df.dropna(axis=1, how="all")

# 3. Clean product names

# Remove unnecessary spaces and convert product names
# to lowercase.
df = df.map(
    lambda x: str(x).strip().lower()
    if pd.notna(x) else pd.NA
)


# 4. Remove duplicate products within transactions

def clean_transaction(row):
    items = []
    seen = set()

    for item in row.dropna():

        # Avoid duplicate items in the same transaction
        if item not in seen:
            items.append(item)
            seen.add(item)

    return pd.Series(items)


cleaned_df = df.apply(clean_transaction, axis=1)


# 5. Give columns meaningful names

cleaned_df.columns = [
    f"item_{i+1}"
    for i in range(cleaned_df.shape[1])
]

# 6. Save the cleaned dataset

output_file = "groceries_cleaned.csv"

cleaned_df.to_csv(
    output_file,
    index=False
)


# 7. Display information about the cleaned dataset

print("\nCleaning completed!")

print("Number of transactions:", len(cleaned_df))

print(
    "Maximum items in one transaction:",
    cleaned_df.notna().sum(axis=1).max()
)

# Get all unique products
unique_products = pd.unique(
    cleaned_df.values.ravel()
)

unique_products = unique_products[
    pd.notna(unique_products)
]

print(
    "Number of unique products:",
    len(unique_products)
)

print("\nFirst 5 transactions:")
print(cleaned_df.head())

print("\nCleaned file saved as:", output_file)

Original shape: (9835, 32)

Cleaning completed!
Number of transactions: 9835
Maximum items in one transaction: 32
Number of unique products: 169

First 5 transactions:
             item_1               item_2          item_3  \
0      citrus fruit  semi-finished bread       margarine   
1    tropical fruit               yogurt          coffee   
2        whole milk                  NaN             NaN   
3         pip fruit               yogurt    cream cheese   
4  other vegetables           whole milk  condensed milk   

                     item_4 item_5 item_6 item_7 item_8 item_9 item_10  ...  \
0               ready soups    NaN    NaN    NaN    NaN    NaN     NaN  ...   
1                       NaN    NaN    NaN    NaN    NaN    NaN     NaN  ...   
2                       NaN    NaN    NaN    NaN    NaN    NaN     NaN  ...   
3              meat spreads    NaN    NaN    NaN    NaN    NaN     NaN  ...   
4  long life bakery product    NaN    NaN    NaN    NaN    NaN     NaN  ... 

### Question 1: Understand The Data

In [3]:
import pandas as pd

# 1. Load the cleaned dataset

df = pd.read_csv("groceries_cleaned.csv")


# a) How many transactions are in the dataset

# Each row represents one transaction
number_of_transactions = len(df)

print("1. Number of transactions:", number_of_transactions)

# b) How many different grocery items are included

# Convert the dataset from columns into one long list
all_items = df.values.flatten()

# Remove missing values
all_items = all_items[pd.notna(all_items)]

# Get unique grocery items
unique_items = pd.unique(all_items)

number_of_unique_items = len(unique_items)

print("2. Number of different grocery items:",
      number_of_unique_items)

# 4. What are the 10 most frequently purchased items

# Count how many times each item appears
item_counts = pd.Series(all_items).value_counts()

# Get the top 10
top_10_items = item_counts.head(10)

print("\n3. Top 10 most frequently purchased items:")
print(top_10_items)

1. Number of transactions: 9835
2. Number of different grocery items: 169

3. Top 10 most frequently purchased items:
whole milk          2513
other vegetables    1903
rolls/buns          1809
soda                1715
yogurt              1372
bottled water       1087
root vegetables     1072
tropical fruit      1032
shopping bags        969
sausage              924
Name: count, dtype: int64


### Question 2: Frequent ItemSet

In [4]:
# FREQUENT ITEMSETS USING APRIORI

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori

# Convert each transaction into a list

transactions = (
    df.apply(lambda row: row.dropna().tolist(), axis=1)
      .tolist()
)


# One-hot encode the transactions

te = TransactionEncoder()

encoded_data = te.fit(transactions).transform(transactions)

basket = pd.DataFrame(
    encoded_data,
    columns=te.columns_
)

print("Transaction matrix shape:", basket.shape)


# a) Find at least 10 frequent itemset containing two or more products

# Minimum support = 1%
# You can adjust this later if needed.
frequent_itemsets = apriori(
    basket,
    min_support=0.01,
    use_colnames=True
)

# Keep only itemsets containing two or more products


frequent_itemsets["itemset_size"] = (
    frequent_itemsets["itemsets"].apply(len)
)

frequent_2plus = frequent_itemsets[
    frequent_itemsets["itemset_size"] >= 2
].copy()

# Report Their Support Values

frequent_2plus = frequent_2plus.sort_values(
    by="support",
    ascending=False
)


Transaction matrix shape: (9835, 169)


In [5]:
# Which itemset has the highest support

top_10_itemsets = frequent_2plus.head(10).copy()


# Convert itemsets to readable text
top_10_itemsets["Itemset"] = (
    top_10_itemsets["itemsets"]
    .apply(lambda x: ", ".join(sorted(x)))
)

In [6]:
# Display the results
print("\nTop 10 Frequent Itemsets:")
print(
    top_10_itemsets[
        ["Itemset", "support"]
    ].to_string(index=False)
)


Top 10 Frequent Itemsets:
                          Itemset  support
     other vegetables, whole milk 0.074835
           rolls/buns, whole milk 0.056634
               whole milk, yogurt 0.056024
      root vegetables, whole milk 0.048907
other vegetables, root vegetables 0.047382
         other vegetables, yogurt 0.043416
     other vegetables, rolls/buns 0.042603
       tropical fruit, whole milk 0.042298
                 soda, whole milk 0.040061
                 rolls/buns, soda 0.038332


In [7]:
# c) Which itemset has the highest support

highest_itemset = frequent_2plus.iloc[0]

print("\nItemset with the highest support:")

print(
    "Itemset:",
    ", ".join(sorted(highest_itemset["itemsets"]))
)

print(
    "Support:",
    round(highest_itemset["support"], 4)
)

print(
    "Support (%):",
    round(highest_itemset["support"] * 100, 2),
    "%"
)


Itemset with the highest support:
Itemset: other vegetables, whole milk
Support: 0.0748
Support (%): 7.48 %


### Question 3: Association Rules

In [8]:
from mlxtend.frequent_patterns import association_rules

# a) Generate association rules

# Generate rules from the frequent itemsets
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.10
)

# b) Select the important columns

rules_result = rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].copy()

# Convert itemsets to readable text

rules_result["Antecedent"] = rules_result[
    "antecedents"
].apply(lambda x: ", ".join(sorted(x)))

rules_result["Consequent"] = rules_result[
    "consequents"
].apply(lambda x: ", ".join(sorted(x)))

# Arrange columns

rules_result = rules_result[
    [
        "Antecedent",
        "Consequent",
        "support",
        "confidence",
        "lift"
    ]
]

# Sort rules by confidence

rules_result = rules_result.sort_values(
    by="confidence",
    ascending=False
)

# Display at least 10 association rules

top_10_rules = rules_result.head(10)

print("\nQUESTION 3: TOP 10 ASSOCIATION RULES")
print("=" * 80)

print(
    top_10_rules.to_string(index=False)
)


QUESTION 3: TOP 10 ASSOCIATION RULES
                     Antecedent       Consequent  support  confidence     lift
  citrus fruit, root vegetables other vegetables 0.010371    0.586207 3.029608
root vegetables, tropical fruit other vegetables 0.012303    0.584541 3.020999
                   curd, yogurt       whole milk 0.010066    0.582353 2.279125
       butter, other vegetables       whole milk 0.011490    0.573604 2.244885
root vegetables, tropical fruit       whole milk 0.011998    0.570048 2.230969
        root vegetables, yogurt       whole milk 0.014540    0.562992 2.203354
domestic eggs, other vegetables       whole milk 0.012303    0.552511 2.162336
     whipped/sour cream, yogurt       whole milk 0.010880    0.524510 2.052747
    rolls/buns, root vegetables       whole milk 0.012710    0.523013 2.046888
    other vegetables, pip fruit       whole milk 0.013523    0.517510 2.025351


### Question 4: Understanding The Measure

In [9]:
# Select 3 interesting rules from our results
three_rules = rules_result.head(3).copy()

print("\nQUESTION 4: THREE ASSOCIATION RULES")
print("=" * 80)

for index, rule in three_rules.iterrows():

    print("\nRule:")
    print(f"{{{rule['Antecedent']}}} → {{{rule['Consequent']}}}")

    print(f"Support: {rule['support']:.4f}")
    print(f"Confidence: {rule['confidence']:.4f}")
    print(f"Lift: {rule['lift']:.4f}")


QUESTION 4: THREE ASSOCIATION RULES

Rule:
{citrus fruit, root vegetables} → {other vegetables}
Support: 0.0104
Confidence: 0.5862
Lift: 3.0296

Rule:
{root vegetables, tropical fruit} → {other vegetables}
Support: 0.0123
Confidence: 0.5845
Lift: 3.0210

Rule:
{curd, yogurt} → {whole milk}
Support: 0.0101
Confidence: 0.5824
Lift: 2.2791


### Question 5: Find Interesting Rule

In [10]:
print("\nQUESTION 5: FIND INTERESTING RULES")
print("=" * 80)

# 1. Rule with the highest confidence
highest_confidence = rules_result.loc[
    rules_result["confidence"].idxmax()
]

print("\n1. RULE WITH HIGHEST CONFIDENCE")
print("-" * 50)
print(
    f"{{{highest_confidence['Antecedent']}}} → "
    f"{{{highest_confidence['Consequent']}}}"
)
print(f"Support: {highest_confidence['support']:.4f}")
print(f"Confidence: {highest_confidence['confidence']:.4f}")
print(f"Lift: {highest_confidence['lift']:.4f}")


# 2. Rule with the highest lift
highest_lift = rules_result.loc[
    rules_result["lift"].idxmax()
]

print("\n2. RULE WITH HIGHEST LIFT")
print("-" * 50)
print(
    f"{{{highest_lift['Antecedent']}}} → "
    f"{{{highest_lift['Consequent']}}}"
)
print(f"Support: {highest_lift['support']:.4f}")
print(f"Confidence: {highest_lift['confidence']:.4f}")
print(f"Lift: {highest_lift['lift']:.4f}")


# 3. Rule with the highest support
highest_support = rules_result.loc[
    rules_result["support"].idxmax()
]

print("\n3. RULE WITH HIGHEST SUPPORT")
print("-" * 50)
print(
    f"{{{highest_support['Antecedent']}}} → "
    f"{{{highest_support['Consequent']}}}"
)
print(f"Support: {highest_support['support']:.4f}")
print(f"Confidence: {highest_support['confidence']:.4f}")
print(f"Lift: {highest_support['lift']:.4f}")


# 4. Check whether they are the same rule
confidence_rule = (
    highest_confidence["Antecedent"],
    highest_confidence["Consequent"]
)

lift_rule = (
    highest_lift["Antecedent"],
    highest_lift["Consequent"]
)

support_rule = (
    highest_support["Antecedent"],
    highest_support["Consequent"]
)

print("\n4. ARE THEY THE SAME RULE?")
print("-" * 50)

if confidence_rule == lift_rule == support_rule:
    print("Yes, all three measures identify the same rule.")
else:
    print("No, the three measures identify different rules.")


QUESTION 5: FIND INTERESTING RULES

1. RULE WITH HIGHEST CONFIDENCE
--------------------------------------------------
{citrus fruit, root vegetables} → {other vegetables}
Support: 0.0104
Confidence: 0.5862
Lift: 3.0296

2. RULE WITH HIGHEST LIFT
--------------------------------------------------
{curd} → {whole milk, yogurt}
Support: 0.0101
Confidence: 0.1889
Lift: 3.3723

3. RULE WITH HIGHEST SUPPORT
--------------------------------------------------
{other vegetables} → {whole milk}
Support: 0.0748
Confidence: 0.3868
Lift: 1.5136

4. ARE THEY THE SAME RULE?
--------------------------------------------------
No, the three measures identify different rules.


### Question 6: Interpret A Rule

In [12]:
# Select the highest-lift rule from Question 5
selected_rule = highest_lift

print("\nQUESTION 6: INTERPRET AN INTERESTING RULE")
print("=" * 80)

print("\nSelected Rule:")
print(
    f"{{{selected_rule['Antecedent']}}} → "
    f"{{{selected_rule['Consequent']}}}"
)

print(f"Support: {selected_rule['support']:.4f}")
print(f"Confidence: {selected_rule['confidence']:.4f}")
print(f"Lift: {selected_rule['lift']:.4f}")

print("\nInterpretation:")
print(
    "Customers who purchased curd had a higher tendency "
    "to also purchase whole milk and yogurt."
)

print(
    f"The rule has a confidence of "
    f"{selected_rule['confidence'] * 100:.2f}%, meaning that "
    f"{selected_rule['confidence'] * 100:.2f}% of transactions "
    f"containing curd also contained whole milk and yogurt."
)

print(
    f"The lift is {selected_rule['lift']:.4f}, which is greater "
    "than 1. Therefore, the products have a positive association."
)

if selected_rule["lift"] > 1:
    print(
        "Based on the lift, this represents a positive association "
        "rather than a negative or independent relationship."
    )


QUESTION 6: INTERPRET AN INTERESTING RULE

Selected Rule:
{curd} → {whole milk, yogurt}
Support: 0.0101
Confidence: 0.1889
Lift: 3.3723

Interpretation:
Customers who purchased curd had a higher tendency to also purchase whole milk and yogurt.
The rule has a confidence of 18.89%, meaning that 18.89% of transactions containing curd also contained whole milk and yogurt.
The lift is 3.3723, which is greater than 1. Therefore, the products have a positive association.
Based on the lift, this represents a positive association rather than a negative or independent relationship.


### Question 7: Business Application

In [13]:
print("\nQUESTION 7: BUSINESS APPLICATION")
print("=" * 80)

# 1. Products to place close together
print("\n1. PRODUCTS TO PLACE CLOSE TOGETHER")
print("-" * 50)
print("Other vegetables and whole milk")
print("Reason: They have the highest-support association:")
print(f"Support = {highest_support['support']:.4f}")
print(f"Confidence = {highest_support['confidence']:.4f}")
print(f"Lift = {highest_support['lift']:.4f}")

# 2. Possible product bundle
print("\n2. POSSIBLE PRODUCT BUNDLE")
print("-" * 50)
print("Curd + Whole milk + Yogurt")
print("Reason: This combination has the highest lift:")
print(f"Support = {highest_lift['support']:.4f}")
print(f"Confidence = {highest_lift['confidence']:.4f}")
print(f"Lift = {highest_lift['lift']:.4f}")

# 3. Product recommendation example
print("\n3. PRODUCT RECOMMENDATION EXAMPLE")
print("-" * 50)
print(
    "If a customer purchases curd, the store could recommend "
    "whole milk and yogurt because the association rule is:"
)
print("{curd} → {whole milk, yogurt}")
print(f"Lift = {highest_lift['lift']:.4f}")


QUESTION 7: BUSINESS APPLICATION

1. PRODUCTS TO PLACE CLOSE TOGETHER
--------------------------------------------------
Other vegetables and whole milk
Reason: They have the highest-support association:
Support = 0.0748
Confidence = 0.3868
Lift = 1.5136

2. POSSIBLE PRODUCT BUNDLE
--------------------------------------------------
Curd + Whole milk + Yogurt
Reason: This combination has the highest lift:
Support = 0.0101
Confidence = 0.1889
Lift = 3.3723

3. PRODUCT RECOMMENDATION EXAMPLE
--------------------------------------------------
If a customer purchases curd, the store could recommend whole milk and yogurt because the association rule is:
{curd} → {whole milk, yogurt}
Lift = 3.3723


### Question 8: Understanding Association

In [14]:
print("\nQUESTION 8: UNDERSTANDING ASSOCIATION")
print("=" * 80)

print("\n1. Does association prove causation?")
print("-" * 50)
print(
    "No. Association does not prove causation. "
    "Association only shows that products tend to occur "
    "together in transactions."
)

print("\n2. Why can a rule have high confidence but low lift?")
print("-" * 50)
print(
    "A rule can have high confidence but low lift when the "
    "consequent is already very common in the dataset. "
    "Therefore, many transactions containing the antecedent "
    "may also contain the consequent even without a strong "
    "additional association."
)

print("\n3. What does lift > 1 indicate?")
print("-" * 50)
print(
    "Lift greater than 1 indicates a positive association. "
    "The consequent is purchased more often when the "
    "antecedent is present than would be expected based "
    "on the consequent's overall frequency."
)


QUESTION 8: UNDERSTANDING ASSOCIATION

1. Does association prove causation?
--------------------------------------------------
No. Association does not prove causation. Association only shows that products tend to occur together in transactions.

2. Why can a rule have high confidence but low lift?
--------------------------------------------------
A rule can have high confidence but low lift when the consequent is already very common in the dataset. Therefore, many transactions containing the antecedent may also contain the consequent even without a strong additional association.

3. What does lift > 1 indicate?
--------------------------------------------------
Lift greater than 1 indicates a positive association. The consequent is purchased more often when the antecedent is present than would be expected based on the consequent's overall frequency.
